# Network Centrality Analysis - Helsinki City Bikes

This notebook analyzes various centrality measures and community detection algorithms for the Helsinki City Bike network using April 2021 data.

## Centrality Measures Covered:
1. **Degree Centrality** - Number of connections
2. **Betweenness Centrality** - Bridge nodes on shortest paths
3. **Closeness Centrality** - Average distance to all nodes
4. **Eigenvector Centrality** - Influence based on important neighbors
5. **Community Detection** - Louvain & Fluid Communities algorithms

## Import Libraries

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import collections
from matplotlib import cm
from pathlib import Path

# Color constants
BLUE = '#1f77b4'
GREEN = '#2ca02c'
RED = '#d62728'

# Plot settings
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 11

%matplotlib inline

print("✓ Libraries loaded successfully!")

## Load Data and Build Network

In [ ]:
# Resolve paths
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / '2021-04_merged.csv'
print(f"Loading data from: {DATA_PATH}")

# Load trip data with coordinates
df = pd.read_csv(DATA_PATH)
df['Departure'] = pd.to_datetime(df['Departure'])
df['Return'] = pd.to_datetime(df['Return'])

print(f"✓ Loaded {len(df):,} trips from April 2021")
print(f"✓ Date range: {df['Departure'].min().date()} to {df['Departure'].max().date()}")

In [ ]:
# Build station master table
stations_dep = df[['Departure station id', 'Departure station name', 'departure_lat', 'departure_lon']].drop_duplicates()
stations_dep.columns = ['station_id', 'station_name', 'lat', 'lon']
stations_ret = df[['Return station id', 'Return station name', 'return_lat', 'return_lon']].drop_duplicates()
stations_ret.columns = ['station_id', 'station_name', 'lat', 'lon']
stations_df = pd.concat([stations_dep, stations_ret], ignore_index=True)
stations_df = stations_df.drop_duplicates(subset='station_id').sort_values('station_id').reset_index(drop=True)

# Aggregate edges
edge_df = (
    df.groupby(['Departure station id', 'Return station id'], as_index=False)
      .agg(trip_count=('Departure station id', 'size'),
           total_distance_m=('Covered distance (m)', 'sum'),
           total_duration_s=('Duration (sec.)', 'sum'))
)

print(f"\n✓ Prepared {len(stations_df)} stations and {len(edge_df):,} directed edges")

In [ ]:
# Create position dictionary from station data
pos_dict = {}
for _, row in stations_df.iterrows():
    if pd.notna(row['lat']) and pd.notna(row['lon']):
        pos_dict[int(row['station_id'])] = (row['lon'], row['lat'])

# Build undirected graph (for most centrality measures)
G = nx.Graph()

# Add nodes with station names and positions
for _, row in stations_df.iterrows():
    station_id = int(row['station_id'])
    station_name = row['station_name']
    G.add_node(station_id, station_name=station_name)
    
    if station_id in pos_dict:
        G.nodes[station_id]['pos'] = pos_dict[station_id]

# Add edges with weights
for _, row in edge_df.iterrows():
    src = int(row['Departure station id'])
    dst = int(row['Return station id'])
    
    if G.has_edge(src, dst):
        G[src][dst]['trip_count'] += int(row['trip_count'])
        G[src][dst]['distance'] += float(row['total_distance_m'])
        G[src][dst]['duration'] += float(row['total_duration_s'])
    else:
        G.add_edge(src, dst,
                   trip_count=int(row['trip_count']),
                   distance=float(row['total_distance_m']),
                   duration=float(row['total_duration_s']))

print(f"\n✓ Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges():,} edges")
print(f"✓ Nodes with coordinates: {sum(1 for n in G.nodes() if 'pos' in G.nodes[n])}")
print(f"✓ Network density: {nx.density(G):.4f}")
print(f"✓ Average degree: {sum(dict(G.degree()).values()) / G.number_of_nodes():.2f}")

---
## 1. Degree Centrality

**Definition:** The fraction of nodes that a given node is connected to.

**Interpretation:** Stations with high degree centrality are major hubs with many direct connections.

In [ ]:
def visualize_degree_centrality(network):
    """
    Visualize degree centrality with enhanced styling.
    """
    fig, ax = plt.subplots(figsize=(24, 14))

    # Calculate degree centrality
    centrality_values = nx.degree_centrality(network)
    node_colors = [centrality_values[node] for node in network.nodes()]
    
    # Scale node sizes based on centrality (min 30, max 600)
    max_cent = max(centrality_values.values())
    node_sizes = [30 + (centrality_values[node] / max_cent) * 570 for node in network.nodes()]

    # Color mapping
    vmin, vmax = min(node_colors), max(node_colors)
    cmap = plt.cm.plasma
    
    # Draw network
    nodes = nx.draw_networkx_nodes(
        network,
        pos=network.nodes.data('pos'),
        node_color=node_colors,
        node_size=node_sizes,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        alpha=0.85,
        edgecolors='black',
        linewidths=0.8,
        ax=ax
    )
    
    nx.draw_networkx_edges(
        network,
        pos=network.nodes.data('pos'),
        edge_color='gray',
        width=0.3,
        alpha=0.12,
        ax=ax
    )
    
    # Highlight and label top 5 stations
    top_5 = sorted(centrality_values.items(), key=lambda x: x[1], reverse=True)[:5]
    top_labels = {node: network.nodes[node].get('station_name', f'Station {node}') 
                  for node, _ in top_5}
    
    nx.draw_networkx_labels(
        network,
        pos=dict(network.nodes.data('pos')),
        labels=top_labels,
        font_size=11,
        font_weight='bold',
        font_color='white',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='purple', alpha=0.8, edgecolor='white', linewidth=1.5),
        ax=ax
    )

    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.02)
    cbar.set_label('Degree Centrality', rotation=270, labelpad=25, fontsize=15, fontweight='bold')
    cbar.ax.tick_params(labelsize=12)

    # Title
    ax.set_title('Degree Centrality - Hub Stations in Helsinki Bike Network',
                fontsize=24, fontweight='bold', pad=25, color='#2c3e50')
    ax.axis('off')
    
    # Statistics box
    avg_cent = np.mean(list(centrality_values.values()))
    max_node = max(centrality_values, key=centrality_values.get)
    max_station = network.nodes[max_node].get('station_name', f'Station {max_node}')
    
    stats_text = f"Network Statistics:\n"
    stats_text += f"━━━━━━━━━━━━━━━━━\n"
    stats_text += f"Stations: {network.number_of_nodes()}\n"
    stats_text += f"Connections: {network.number_of_edges():,}\n"
    stats_text += f"Avg Centrality: {avg_cent:.4f}\n"
    stats_text += f"Max Centrality: {max(centrality_values.values()):.4f}\n\n"
    stats_text += f"Top Hub Station:\n{max_station}"
    
    props = dict(boxstyle='round,pad=0.8', facecolor='white', alpha=0.92, 
                edgecolor='purple', linewidth=2.5)
    ax.text(0.015, 0.985, stats_text, transform=ax.transAxes, fontsize=12,
           verticalalignment='top', bbox=props, family='monospace')
    
    plt.tight_layout()
    plt.show()
    
    # Print top 10
    print("\n" + "="*60)
    print("TOP 10 STATIONS BY DEGREE CENTRALITY")
    print("="*60)
    for i, (node, cent) in enumerate(sorted(centrality_values.items(), 
                                            key=lambda x: x[1], reverse=True)[:10], 1):
        name = network.nodes[node].get('station_name', f'Station {node}')
        degree = network.degree(node)
        print(f"{i:2d}. {name:30s} | Centrality: {cent:.4f} | Degree: {degree}")
    print("="*60)


visualize_degree_centrality(G)

## 2. Degree Distribution Analysis

In [ ]:
def plot_degree_distribution(network):
    """
    Plot degree distribution with histogram and cumulative distribution.
    """
    degrees = [deg for node, deg in network.degree()]
    degree_freq = collections.Counter(degrees)
    degree_vals, counts = zip(*sorted(degree_freq.items()))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 9))
    
    # === LEFT: Degree Distribution Histogram ===
    bars = ax1.bar(degree_vals, counts, width=1.0, color=BLUE, alpha=0.75, 
                   edgecolor='navy', linewidth=0.8)
    
    # Highlight tallest bar
    max_idx = counts.index(max(counts))
    bars[max_idx].set_color('orange')
    bars[max_idx].set_alpha(0.9)
    
    mean_deg = np.mean(degrees)
    median_deg = np.median(degrees)
    
    ax1.axvline(mean_deg, color=GREEN, linestyle='--', linewidth=3, 
               label=f'Mean: {mean_deg:.1f}', alpha=0.8)
    ax1.axvline(median_deg, color=RED, linestyle='--', linewidth=3, 
               label=f'Median: {median_deg:.0f}', alpha=0.8)

    ax1.set_title('Degree Distribution', fontsize=20, fontweight='bold', pad=15)
    ax1.set_xlabel('Degree (Number of Connections)', fontsize=15, fontweight='bold')
    ax1.set_ylabel('Frequency (Number of Stations)', fontsize=15, fontweight='bold')
    ax1.legend(fontsize=14, loc='upper right', framealpha=0.95)
    ax1.grid(True, alpha=0.25, linestyle='--', linewidth=0.8)
    ax1.tick_params(labelsize=13)
    ax1.set_axisbelow(True)
    
    # === RIGHT: Cumulative Distribution ===
    sorted_degrees = sorted(degrees, reverse=True)
    cumulative = np.arange(1, len(sorted_degrees) + 1) / len(sorted_degrees)
    
    ax2.plot(sorted_degrees, cumulative, linewidth=3.5, color='darkblue', 
            marker='o', markersize=5, markevery=max(1, len(sorted_degrees)//30),
            alpha=0.8, markerfacecolor='orange', markeredgecolor='navy', markeredgewidth=1.5)
    
    ax2.fill_between(sorted_degrees, cumulative, alpha=0.2, color='blue')
    
    ax2.set_title('Cumulative Degree Distribution', fontsize=20, fontweight='bold', pad=15)
    ax2.set_xlabel('Degree (Number of Connections)', fontsize=15, fontweight='bold')
    ax2.set_ylabel('Cumulative Probability', fontsize=15, fontweight='bold')
    ax2.grid(True, alpha=0.25, linestyle='--', linewidth=0.8)
    ax2.tick_params(labelsize=13)
    ax2.set_axisbelow(True)
    
    # Statistics box
    stats = f"Statistics:\n"
    stats += f"━━━━━━━━━━━━━\n"
    stats += f"Min: {min(degrees)}\n"
    stats += f"Max: {max(degrees)}\n"
    stats += f"Mean: {mean_deg:.2f}\n"
    stats += f"Median: {median_deg:.0f}\n"
    stats += f"Std Dev: {np.std(degrees):.2f}"
    
    props = dict(boxstyle='round,pad=0.7', facecolor='lightyellow', alpha=0.9,
                edgecolor='orange', linewidth=2)
    ax2.text(0.97, 0.05, stats, transform=ax2.transAxes, fontsize=12,
            verticalalignment='bottom', horizontalalignment='right', 
            bbox=props, family='monospace')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nDegree Distribution Summary:")
    print(f"  Range: {min(degrees)} - {max(degrees)}")
    print(f"  Mean ± Std: {mean_deg:.2f} ± {np.std(degrees):.2f}")
    print(f"  Most common degree: {degree_vals[max_idx]} ({max(counts)} stations)")


plot_degree_distribution(G)

---
## 3. Betweenness Centrality

**Definition:** The fraction of shortest paths between all pairs of nodes that pass through a given node.

**Interpretation:** Stations with high betweenness are critical bridges that connect different parts of the network.

In [ ]:
def visualize_betweenness(network):
    """
    Visualize betweenness centrality - identifies bridge stations.
    """
    fig, ax = plt.subplots(figsize=(24, 14))

    print("Calculating betweenness centrality (this may take a moment)...")
    betweenness = nx.betweenness_centrality(network)
    
    node_colors = [betweenness[node] for node in network.nodes()]
    max_between = max(betweenness.values())
    node_sizes = [30 + (betweenness[node] / max_between) * 570 for node in network.nodes()]

    vmin, vmax = min(node_colors), max(node_colors)
    cmap = plt.cm.YlOrRd
    
    nx.draw_networkx_nodes(
        network,
        pos=network.nodes.data('pos'),
        node_color=node_colors,
        node_size=node_sizes,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        alpha=0.85,
        edgecolors='black',
        linewidths=0.8,
        ax=ax
    )
    
    nx.draw_networkx_edges(
        network,
        pos=network.nodes.data('pos'),
        edge_color='gray',
        width=0.3,
        alpha=0.12,
        ax=ax
    )

    # Label top 5 bridge stations
    top_5 = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:5]
    labels = {node: network.nodes[node].get('station_name', f'Station {node}') 
              for node, _ in top_5}
    
    nx.draw_networkx_labels(
        network,
        pos=dict(network.nodes.data('pos')),
        labels=labels,
        font_size=11,
        font_weight='bold',
        font_color='white',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='darkred', alpha=0.85, 
                 edgecolor='white', linewidth=1.5),
        ax=ax
    )

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.02)
    cbar.set_label('Betweenness Centrality', rotation=270, labelpad=25, 
                  fontsize=15, fontweight='bold')
    cbar.ax.tick_params(labelsize=12)

    ax.set_title('Betweenness Centrality - Critical Bridge Stations',
                fontsize=24, fontweight='bold', pad=25, color='#2c3e50')
    ax.axis('off')
    
    # Info box
    info = f"Top 5 Bridge Stations:\n"
    info += f"━━━━━━━━━━━━━━━━━━━━━━━━━\n"
    for i, (node, cent) in enumerate(top_5, 1):
        name = network.nodes[node].get('station_name', f'Station {node}')[:22]
        info += f"{i}. {name}\n   ({cent:.5f})\n"
    
    props = dict(boxstyle='round,pad=0.8', facecolor='white', alpha=0.92,
                edgecolor='darkred', linewidth=2.5)
    ax.text(0.015, 0.985, info, transform=ax.transAxes, fontsize=11,
           verticalalignment='top', bbox=props, family='monospace')
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*60)
    print("TOP 10 BRIDGE STATIONS (BETWEENNESS CENTRALITY)")
    print("="*60)
    for i, (node, cent) in enumerate(sorted(betweenness.items(), 
                                            key=lambda x: x[1], reverse=True)[:10], 1):
        name = network.nodes[node].get('station_name', f'Station {node}')
        print(f"{i:2d}. {name:30s} | Betweenness: {cent:.6f}")
    print("="*60)


visualize_betweenness(G)

---
## 4. Closeness Centrality

**Definition:** The reciprocal of the average shortest path distance from a node to all other nodes.

**Interpretation:** Stations with high closeness centrality are most accessible from all other stations.

In [ ]:
def visualize_closeness(network):
    """
    Visualize closeness centrality - most accessible stations.
    """
    fig, ax = plt.subplots(figsize=(24, 14))

    print("Calculating closeness centrality...")
    closeness = nx.closeness_centrality(network)
    
    node_colors = [closeness[node] for node in network.nodes()]
    max_close = max(closeness.values())
    node_sizes = [30 + (closeness[node] / max_close) * 570 for node in network.nodes()]

    vmin, vmax = min(node_colors), max(node_colors)
    cmap = plt.cm.viridis
    
    nx.draw_networkx_nodes(
        network,
        pos=network.nodes.data('pos'),
        node_color=node_colors,
        node_size=node_sizes,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        alpha=0.85,
        edgecolors='black',
        linewidths=0.8,
        ax=ax
    )
    
    nx.draw_networkx_edges(
        network,
        pos=network.nodes.data('pos'),
        edge_color='gray',
        width=0.3,
        alpha=0.12,
        ax=ax
    )

    # Label top 5 most accessible
    top_5 = sorted(closeness.items(), key=lambda x: x[1], reverse=True)[:5]
    labels = {node: network.nodes[node].get('station_name', f'Station {node}') 
              for node, _ in top_5}
    
    nx.draw_networkx_labels(
        network,
        pos=dict(network.nodes.data('pos')),
        labels=labels,
        font_size=11,
        font_weight='bold',
        font_color='white',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='darkgreen', alpha=0.85,
                 edgecolor='white', linewidth=1.5),
        ax=ax
    )

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.02)
    cbar.set_label('Closeness Centrality', rotation=270, labelpad=25,
                  fontsize=15, fontweight='bold')
    cbar.ax.tick_params(labelsize=12)

    ax.set_title('Closeness Centrality - Most Accessible Stations',
                fontsize=24, fontweight='bold', pad=25, color='#2c3e50')
    ax.axis('off')
    
    # Info box
    info = f"Top 5 Most Central Stations:\n"
    info += f"━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
    for i, (node, cent) in enumerate(top_5, 1):
        name = network.nodes[node].get('station_name', f'Station {node}')[:22]
        info += f"{i}. {name}\n   ({cent:.5f})\n"
    
    props = dict(boxstyle='round,pad=0.8', facecolor='white', alpha=0.92,
                edgecolor='darkgreen', linewidth=2.5)
    ax.text(0.015, 0.985, info, transform=ax.transAxes, fontsize=11,
           verticalalignment='top', bbox=props, family='monospace')
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*60)
    print("TOP 10 MOST ACCESSIBLE STATIONS (CLOSENESS CENTRALITY)")
    print("="*60)
    for i, (node, cent) in enumerate(sorted(closeness.items(),
                                            key=lambda x: x[1], reverse=True)[:10], 1):
        name = network.nodes[node].get('station_name', f'Station {node}')
        print(f"{i:2d}. {name:30s} | Closeness: {cent:.6f}")
    print("="*60)


visualize_closeness(G)

---
## 5. Eigenvector Centrality

**Definition:** A measure of influence based on connections to other influential nodes.

**Interpretation:** Stations connected to other important stations have high eigenvector centrality.

In [ ]:
def visualize_eigenvector(network):
    """
    Visualize eigenvector centrality - influence through important connections.
    """
    fig, ax = plt.subplots(figsize=(24, 14))

    print("Calculating eigenvector centrality...")
    eigenvector = nx.eigenvector_centrality(network, max_iter=6000, weight='duration')
    
    node_colors = [eigenvector[node] for node in network.nodes()]
    max_eigen = max(eigenvector.values())
    node_sizes = [30 + (eigenvector[node] / max_eigen) * 570 for node in network.nodes()]

    vmin, vmax = min(node_colors), max(node_colors)
    cmap = plt.cm.coolwarm
    
    nx.draw_networkx_nodes(
        network,
        pos=network.nodes.data('pos'),
        node_color=node_colors,
        node_size=node_sizes,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        alpha=0.85,
        edgecolors='black',
        linewidths=0.8,
        ax=ax
    )
    
    nx.draw_networkx_edges(
        network,
        pos=network.nodes.data('pos'),
        edge_color='gray',
        width=0.3,
        alpha=0.12,
        ax=ax
    )

    # Label top 5 influential
    top_5 = sorted(eigenvector.items(), key=lambda x: x[1], reverse=True)[:5]
    labels = {node: network.nodes[node].get('station_name', f'Station {node}') 
              for node, _ in top_5}
    
    nx.draw_networkx_labels(
        network,
        pos=dict(network.nodes.data('pos')),
        labels=labels,
        font_size=11,
        font_weight='bold',
        font_color='white',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='darkblue', alpha=0.85,
                 edgecolor='white', linewidth=1.5),
        ax=ax
    )

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.02)
    cbar.set_label('Eigenvector Centrality', rotation=270, labelpad=25,
                  fontsize=15, fontweight='bold')
    cbar.ax.tick_params(labelsize=12)

    ax.set_title('Eigenvector Centrality - Most Influential Stations',
                fontsize=24, fontweight='bold', pad=25, color='#2c3e50')
    ax.axis('off')
    
    # Info box
    info = f"Top 5 Influential Stations:\n"
    info += f"━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
    for i, (node, cent) in enumerate(top_5, 1):
        name = network.nodes[node].get('station_name', f'Station {node}')[:22]
        info += f"{i}. {name}\n   ({cent:.5f})\n"
    
    props = dict(boxstyle='round,pad=0.8', facecolor='white', alpha=0.92,
                edgecolor='darkblue', linewidth=2.5)
    ax.text(0.015, 0.985, info, transform=ax.transAxes, fontsize=11,
           verticalalignment='top', bbox=props, family='monospace')
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*60)
    print("TOP 10 MOST INFLUENTIAL STATIONS (EIGENVECTOR CENTRALITY)")
    print("="*60)
    for i, (node, cent) in enumerate(sorted(eigenvector.items(),
                                            key=lambda x: x[1], reverse=True)[:10], 1):
        name = network.nodes[node].get('station_name', f'Station {node}')
        print(f"{i:2d}. {name:30s} | Eigenvector: {cent:.6f}")
    print("="*60)


visualize_eigenvector(G)

---
## 6. Community Detection - Louvain Algorithm

The Louvain method detects communities by optimizing modularity - groups of nodes that are more densely connected internally than externally.

In [ ]:
def detect_communities_louvain(network):
    """
    Louvain community detection with enhanced visualization.
    """
    fig, ax = plt.subplots(figsize=(24, 14))

    print("Detecting communities using Louvain algorithm...")
    from networkx.algorithms import community as nx_comm
    
    communities_list = nx_comm.louvain_communities(network, seed=1, resolution=0.95)
    
    # Node to community mapping
    communities = {}
    for idx, comm_set in enumerate(communities_list):
        for node in comm_set:
            communities[node] = idx

    num_communities = len(communities_list)
    
    # Colormap
    if num_communities <= 20:
        cmap = cm.get_cmap('tab20', num_communities)
    else:
        cmap = cm.get_cmap('hsv', num_communities)

    # Node sizes based on community size
    comm_sizes = {i: len(comm) for i, comm in enumerate(communities_list)}
    node_sizes = [40 + comm_sizes[communities[node]] * 3 for node in network.nodes()]
    node_colors = [communities[node] for node in network.nodes()]

    nx.draw_networkx_nodes(
        network,
        pos=network.nodes.data('pos'),
        node_color=node_colors,
        node_size=node_sizes,
        cmap=cmap,
        alpha=0.85,
        edgecolors='black',
        linewidths=0.8,
        ax=ax
    )
    
    nx.draw_networkx_edges(
        network,
        pos=network.nodes.data('pos'),
        edge_color='gray',
        width=0.2,
        alpha=0.08,
        ax=ax
    )

    ax.set_title(f'Community Detection - Louvain Method ({num_communities} Communities)',
                fontsize=24, fontweight='bold', pad=25, color='#2c3e50')
    ax.axis('off')
    
    # Statistics
    sizes = sorted([len(c) for c in communities_list], reverse=True)
    
    info = f"Community Statistics:\n"
    info += f"━━━━━━━━━━━━━━━━━━━━━\n"
    info += f"Total: {num_communities} communities\n"
    info += f"Largest: {sizes[0]} stations\n"
    info += f"Smallest: {sizes[-1]} stations\n"
    info += f"Average: {np.mean(sizes):.1f}\n"
    info += f"Median: {np.median(sizes):.0f}\n\n"
    info += f"Top 5 Sizes:\n"
    for i, size in enumerate(sizes[:5], 1):
        info += f"{i}. {size} stations\n"
    
    props = dict(boxstyle='round,pad=0.8', facecolor='white', alpha=0.92,
                edgecolor='purple', linewidth=2.5)
    ax.text(0.015, 0.985, info, transform=ax.transAxes, fontsize=12,
           verticalalignment='top', bbox=props, family='monospace')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✓ Detected {num_communities} communities")
    print(f"✓ Size distribution: {sizes[:10]}...")
    print(f"✓ Modularity optimized for best community structure")


detect_communities_louvain(G)

---
## 7. Community Detection - Fluid Communities

Fluid Communities algorithm propagates labels through the network like a fluid, grouping nodes into k communities.

In [ ]:
def detect_communities_fluid(network, k=5):
    """
    Fluid communities detection with enhanced visualization.
    """
    fig, ax = plt.subplots(figsize=(24, 14))

    print(f"Detecting {k} communities using Fluid Communities algorithm...")
    
    community_sets = nx.algorithms.community.asyn_fluid.asyn_fluidc(
        network, k=k, max_iter=10000, seed=1
    )

    communities = {}
    communities_list = []
    for cluster_id, nodes in enumerate(community_sets):
        communities_list.append(nodes)
        for node in nodes:
            communities[node] = cluster_id

    num_detected = len(communities_list)
    cmap = cm.get_cmap('Set3', num_detected)

    # Node sizes based on community
    comm_sizes = {i: len(comm) for i, comm in enumerate(communities_list)}
    node_sizes = [40 + comm_sizes[communities[node]] * 3 for node in network.nodes()]
    node_colors = [communities[node] for node in network.nodes()]

    nx.draw_networkx_nodes(
        network,
        pos=network.nodes.data('pos'),
        node_color=node_colors,
        node_size=node_sizes,
        cmap=cmap,
        alpha=0.85,
        edgecolors='black',
        linewidths=0.8,
        ax=ax
    )
    
    nx.draw_networkx_edges(
        network,
        pos=network.nodes.data('pos'),
        edge_color='gray',
        width=0.2,
        alpha=0.08,
        ax=ax
    )

    ax.set_title(f'Community Detection - Fluid Communities (k={k})',
                fontsize=24, fontweight='bold', pad=25, color='#2c3e50')
    ax.axis('off')
    
    # Statistics
    sizes = sorted([len(c) for c in communities_list], reverse=True)
    
    info = f"Fluid Communities:\n"
    info += f"━━━━━━━━━━━━━━━━━━━━━\n"
    info += f"Requested: {k}\n"
    info += f"Detected: {num_detected}\n"
    info += f"Largest: {sizes[0]} stations\n"
    info += f"Smallest: {sizes[-1]} stations\n"
    info += f"Average: {np.mean(sizes):.1f}\n\n"
    info += f"All Sizes:\n"
    for i, size in enumerate(sizes, 1):
        info += f"{i}. {size} stations\n"
    
    props = dict(boxstyle='round,pad=0.8', facecolor='white', alpha=0.92,
                edgecolor='orange', linewidth=2.5)
    ax.text(0.015, 0.985, info, transform=ax.transAxes, fontsize=12,
           verticalalignment='top', bbox=props, family='monospace')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✓ Detected {num_detected} communities")
    print(f"✓ Community balance: {sizes}")


detect_communities_fluid(G, k=5)

---
## Summary

This notebook analyzed the Helsinki City Bike network using various centrality measures:

- **Degree Centrality**: Identified hub stations with the most connections
- **Betweenness Centrality**: Found critical bridge stations connecting different network parts
- **Closeness Centrality**: Located the most accessible stations
- **Eigenvector Centrality**: Determined influential stations based on network structure
- **Community Detection**: Discovered natural groupings of stations using Louvain and Fluid Communities algorithms

These metrics provide insights into network structure, station importance, and community organization in Helsinki's bike-sharing system.